# Paso 1

#### Librerias y lectura

In [28]:
import json
import re
import pandas as pd
import unicodedata
from thefuzz import fuzz

In [29]:
df = pd.read_json('catalogo_tp3.jsonl', lines=True)

#### Vista inicial

In [30]:
total_inicial = len(df)
print(f"Total de registros iniciales: {total_inicial}")

Total de registros iniciales: 8747


In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   source_url      8747 non-null   str                
 1   slug            8747 non-null   str                
 2   title           8747 non-null   str                
 3   authors         8747 non-null   object             
 4   genres          8747 non-null   object             
 5   seed_genre      8747 non-null   str                
 6   series          2316 non-null   str                
 7   series_number   2298 non-null   float64            
 8   cover_url       8747 non-null   str                
 9   synopsis        8746 non-null   str                
 10  synopsis_chars  8747 non-null   int64              
 11  scraped_at      8747 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(1), object(2), str(7)
memory usage: 820.2+ KB


In [32]:
df.head()

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


#### Analisis de sinopsis

In [33]:
(df['synopsis'] == "").sum()

np.int64(0)

No hay sinopsis vacias

In [34]:
# detectar filas con sinapsis = a NaN
df[df['synopsis'].isnull()]

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
7969,https://ww3.lectulandia.com/book/calvario/,calvario,Calvario,[Alejandro Sawa],"[Drama, Teatro]",teatro,NaN,NaN,https://assets.lectulandia.com/b/ab/Alejandro%...,NaN,0,2026-10-18 22:48:58+00:00


Hay una sola fila con sinapsis vacia

In [35]:
df = df.dropna(subset=["synopsis"])
df = df[df["synopsis"].astype(str).str.strip() != ""]

In [36]:
# detectar cuantas y eliminar las filas con slug duplicados
df.duplicated(subset=['slug']).sum()

np.int64(0)

No hay libros duplicados

In [37]:
df = df.drop_duplicates(subset=["slug"], keep="first")

In [38]:
len(df)

8746

In [39]:
# Normalizar espacios en blanco (reemplazar saltos de línea y tabs por un único espacio)
antes = len(df)
df["synopsis"] = df["synopsis"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())
df["title"] = df["title"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())

In [40]:
# Contar duplicados exactos de sinopsis
sinopsis_duplicadas = df[df.duplicated(subset=["synopsis"], keep=False)]
print(f"Registros con sinopsis iguales: {len(sinopsis_duplicadas)}")

Registros con sinopsis iguales: 131


In [41]:
# Conservar una sola versión por sinopsis
df = df.drop_duplicates(subset=["synopsis"], keep="first")
eliminados_sinopsis = antes - len(df)
print(f"Libros eliminados por sinopsis iguales: {eliminados_sinopsis}")

Libros eliminados por sinopsis iguales: 98


#### Busqueda de libros "casi iguales"

Normalizacion de los titulos

In [50]:
def normalizar_titulo(texto):
    if pd.isna(texto):
        return ""

    texto = texto.lower()

    # Quitar acentos
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    # Eliminar puntuación
    texto = re.sub(r"[^\w\s]", " ", texto)

    # Espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


df["title_norm"] = df["title"].apply(normalizar_titulo)

Buscar candidatos

In [51]:
# Columna auxiliar para agrupar por autor
df["authors_norm"] = df["authors"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)

pares = []

romanos = r"\b(i|ii|iii|iv|v|vi|vii|viii|ix|x)\b"

for autor, grupo in df.groupby("authors_norm"):

    grupo = grupo.reset_index()

    if len(grupo) < 2:
        continue

    for i in range(len(grupo)):
        for j in range(i + 1, len(grupo)):

            titulo1 = grupo.loc[i, "title_norm"]
            titulo2 = grupo.loc[j, "title_norm"]

            score = fuzz.token_set_ratio(titulo1, titulo2)

            if score < 90:
                continue

            # Si tienen números distintos (años, tomos, etc.)
            nums1 = re.findall(r"\d+", titulo1)
            nums2 = re.findall(r"\d+", titulo2)

            if nums1 != nums2:
                continue

            # Si cambian los números romanos
            rom1 = re.findall(romanos, titulo1)
            rom2 = re.findall(romanos, titulo2)

            if rom1 != rom2:
                continue

            # Evitar tomos o volúmenes diferentes
            palabras = ["vol", "volumen", "tomo", "parte", "libro", "numero"]

            if any(p in titulo1 or p in titulo2 for p in palabras):
                if titulo1 != titulo2:
                    continue

            pares.append({
                "idx1": grupo.loc[i, "index"],
                "idx2": grupo.loc[j, "index"],
                "autor": autor,
                "titulo1": grupo.loc[i, "title"],
                "titulo2": grupo.loc[j, "title"],
                "score_titulo": score
            })

pares = pd.DataFrame(pares)

print(f"Pares candidatos encontrados: {len(pares)}")
display(pares.head())

Pares candidatos encontrados: 110


,idx1,idx2,autor,titulo1,titulo2,score_titulo
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100


Comparar sinopsis

In [54]:
pares["score_sinopsis"] = pares.apply(
    lambda fila: fuzz.token_set_ratio(
        df.loc[fila["idx1"], "synopsis"],
        df.loc[fila["idx2"], "synopsis"]
    ),
    axis=1
)

display(pares.head())

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100,59
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100,58
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100,51
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100,58
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100,59


Elegir las posibles "obras iguales" y mostrarlas

In [ ]:
obras_iguales = pares[
    pares["score_sinopsis"] >= 80
].sort_values(
    ["score_titulo", "score_sinopsis"],
    ascending=False
)

print(f"Posibles obras iguales: {len(obras_iguales)}")

Posibles obras iguales: 25


,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
7,1309,1504,Arthur Conan Doyle,Sherlock Holmes,Sherlock Holmes. La colección completa,100,100
37,4862,5876,H. Rider Haggard,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,100,100
43,206,6001,Isaac Asimov,El secreto del universo,El secreto del universo y otros ensayos cientí...,100,100
53,208,6218,John Gribbin,Introducción a la ciencia. Una guía para todos...,Introducción a la ciencia,100,100
61,1550,1803,Lewis Wallace,Ben-Hur,Ben-Hur (Traducción Raimundo Griñó),100,100
64,2065,8462,Mark Twain,El príncipe y el mendigo,Príncipe y mendigo,100,100
65,736,739,Maurice Nicoll,Comentarios psicológicos sobre las enseñanzas ...,Comentarios psicológicos sobre las enseñanzas ...,100,100
82,8028,8032,Richard Ford,Manual para viajeros por Andalucía y lectores ...,Manual para viajeros por Andalucía y lectores ...,100,100
49,359,7456,Jean-Jacques Rousseau,El contrato social,Contrato social,100,99
60,801,5290,Laura Gallego García,La emperatriz de los Etéreos (Ed. Ilustrada),La emperatriz de los Etéreos,100,99


Mostramos estas obras

In [57]:
display(
    obras_iguales[
        [
            "idx1",
            "idx2",
            "autor",
            "titulo1",
            "titulo2",
            "score_titulo",
            "score_sinopsis"
        ]
    ]
)

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
7,1309,1504,Arthur Conan Doyle,Sherlock Holmes,Sherlock Holmes. La colección completa,100,100
37,4862,5876,H. Rider Haggard,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,100,100
43,206,6001,Isaac Asimov,El secreto del universo,El secreto del universo y otros ensayos cientí...,100,100
53,208,6218,John Gribbin,Introducción a la ciencia. Una guía para todos...,Introducción a la ciencia,100,100
61,1550,1803,Lewis Wallace,Ben-Hur,Ben-Hur (Traducción Raimundo Griñó),100,100
64,2065,8462,Mark Twain,El príncipe y el mendigo,Príncipe y mendigo,100,100
65,736,739,Maurice Nicoll,Comentarios psicológicos sobre las enseñanzas ...,Comentarios psicológicos sobre las enseñanzas ...,100,100
82,8028,8032,Richard Ford,Manual para viajeros por Andalucía y lectores ...,Manual para viajeros por Andalucía y lectores ...,100,100
49,359,7456,Jean-Jacques Rousseau,El contrato social,Contrato social,100,99
60,801,5290,Laura Gallego García,La emperatriz de los Etéreos (Ed. Ilustrada),La emperatriz de los Etéreos,100,99
